# Ejercicio 5 (OPCIONAL) — Actualización de la “tabla de escrituras” 

Este punto **no consiste en “alterar tablas” con SQL/CQL**, sino en **modificar la información que el juego envía al servidor** cuando ocurren los eventos de escritura.  
En Cassandra el diseño es mediante **querys**: las tablas se crean para responder a lecturas concretas, y normalmente están **denormalizadas** (duplican datos) porque **no hay JOINs** y hacer lecturas extra en cada escritura empeora la latencia y complica la consistencia.

En nuestro diseño (Tarea 1) las tablas clave son:
- `hall_of_fame_by_country`  
  `PRIMARY KEY ((country, dungeon_id), time_minutes, email)`  
  y además `user_name`, `date`, `dungeon_name STATIC`
- `user_statistics_by_dungeon`  
  `PRIMARY KEY ((email, dungeon_id), time_minutes, date)`
- `top_horde_by_event`  
  `PRIMARY KEY ((country, event_id), n_killed, email)`  
  y además `user_name`

### Por tanto, las escrituras del enunciado original **no son suficientes** para escribir directamente en estas tablas sin hacer lecturas extra.

## Escrituras actualizadas 

### When: User finish dungeon
**Antes (enunciado):**
- `dungeon_id: int`
- `email: str`
- `time_minutes: float`
- `date: str` (ISO 8601)

**Después (adaptado a nuestras tablas Cassandra):**
- `country: str` 
  Necesario para escribir en `hall_of_fame_by_country`, cuya partición es `(country, dungeon_id)`.
- `dungeon_id: int`
- `dungeon_name: str` 
  Se devuelve en el “Hall of Fame” y se guarda como `STATIC` en `hall_of_fame_by_country`.  
  Sin este campo, el servidor tendría que consultar la tabla relacional `Dungeon` (o una tabla Cassandra equivalente).
- `email: str` (identificador del usuario en nuestro diseño)
- `user_name: str` 
  Se devuelve en el “Hall of Fame” (Top 5), y se guarda denormalizado para evitar consultar `WebUser`.
- `time_minutes: int/float`
- `date: str` (ISO 8601)

Aunque exista la tabla `dungeons_by_country`, esta solo guarda ids; no aporta `dungeon_name`, por eso es útil que el evento lo envíe directamente.



### When: User kills monster during Horde event
**Antes (enunciado):**
- `event_id: int`
- `email: str`
- `monster_id: int`

**Después (adaptado a nuestras tablas Cassandra):**
- `country: str` 
  Necesario porque `top_horde_by_event` particiona por `(country, event_id)` y el ranking de horda es local por país.
- `event_id: int`
- `email: str` (identificador del usuario)
- `user_name: str` 
  Se devuelve en la lectura del Top Horde, y se guarda denormalizado para evitar leer `WebUser`.
- `monster_id: int` *(opcional)*  
  Para el leaderboard no es imprescindible (solo necesitamos contar kills), pero puede mantenerse para logging.



### Justificación de nuetsro diseño
1. **Los leaderboards son locales por país**, y nuestro diseño Cassandra usa `country` como parte de la **clave de partición** en `hall_of_fame_by_country` y `top_horde_by_event`. Si `country` no llega en la escritura, no se puede escribir en la partición correcta sin consultar la tabla de usuarios.
2. En Cassandra **no hay JOINs**, y queremos que cada evento de escritura se traduzca en **escrituras directas** sobre las tablas denormalizadas. Si faltan `country`, `user_name` o `dungeon_name`, el backend tendría que hacer lecturas previas para obtenerlos (p.ej., mirar `WebUser.country/userName` o `Dungeon.name`).
3. Guardamos `user_name` y `dungeon_name` denormalizados porque **aparecen en los outputs** de las lecturas. Así, cuando el cliente pide el leaderboard, Cassandra ya tiene toda la información lista y no necesita combinar tablas.
4. `dungeon_name` se almacena como **STATIC** en `hall_of_fame_by_country` para no repetir el mismo nombre en cada fila del Top-5: queda guardado una sola vez por partición `(country, dungeon_id)`.
5. Para las Hordas, el enunciado dice que la latencia durante gameplay es crítica. Evitar lecturas adicionales por cada kill es esencial; con `country` y `user_name` en el evento, la actualización del ranking es inmediata.


# CQL de escrituras usando el nuevo cuerpo de la petición 

## When: User finish dungeon

### Insert en `hall_of_fame_by_country`
```sql
INSERT INTO hall_of_fame_by_country (
  country, dungeon_id, time_minutes, email, user_name, date, dungeon_name
) VALUES (
  :country, :dungeon_id, :time_minutes, :email, :user_name, :date, :dungeon_name
);


### Insert en `user_statistics_by_dungeon`
```sql
INSERT INTO user_statistics_by_dungeon (
  email, dungeon_id, time_minutes, date
) VALUES (
  :email, :dungeon_id, :time_minutes, :date
);

date es TIMESTAMP en nuestras tablas, así que el backend debe convertir ISO 8601 a timestamp (o enviarlo ya como timestamp compatible)

## When: User kills monster during Horde event

### Importante sobre nuestra tabla top_horde_by_event
* Esta tabla está ordenada por n_killed DESC y n_killed forma parte de la clave primaria.
Eso significa que cuando cambie n_killed, no podemos hacer UPDATE de ese valor en la misma fila, ya que en Cassandra cambiar un componente de la clave equivale a insertar una nueva fila con otra clave (y la antigua seguiría existiendo si no la borrasemos).
Por tanto, podemos analizar 2 enfoques:


### Enfoque 1: el backend calcula n_killed y reinserta

1) El backend mantiene el contador (en memoria) y calcula new_n_killed.
2) Inserta una nueva fila en top_horde_by_event con el new_n_killed.

```sql
INSERT INTO top_horde_by_event (
  country, event_id, n_killed, user_name, email
) VALUES (
  :country, :event_id, :new_n_killed, :user_name, :email
);

Si el backend conoce el valor anterior old_n_killed, se puede borrar la fila antigua para evitar duplicados:

```sql
DELETE FROM top_horde_by_event
WHERE country=:country AND event_id=:event_id AND n_killed=:old_n_killed AND email=:email;

La lectura del Top-K se obtiene directamente con:
SELECT ... FROM top_horde_by_event WHERE country=? AND event_id=? LIMIT K;
(gracias al orden n_killed DESC).

### Enfoque 2 (más correcto para tiempo real): añadir una tabla COUNTER

Esto ya sería tocar el diseño. Se podría crear una tabla counter para contar kills por usuario,
y luego construir el Top-K a partir de ahí.

* Elegimos el enfoque 1 (reinserción + delete opcional) porque mantuvimos el diseño original de la tabla top_horde_by_event, donde n_killed forma parte de la clave primaria (PRIMARY KEY ((country, event_id), n_killed, email)). 
* En Cassandra no se puede actualizar un valor que está en la clave primaria, así que cuando cambia n_killed la forma correcta es insertar una nueva fila con el nuevo n_killed (y, si conocemos el anterior, lo borramos). Además, este enfoque permite seguir leyendo el Top-K directamente con LIMIT K gracias al orden n_killed DESC sin añadir tablas nuevas.